In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from collections import Counter
from itertools import combinations
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
warnings.filterwarnings("ignore")

In [33]:
# Load data
df = pd.read_csv("train_clean.csv")

In [34]:
# Convert the date column to datetime
df["capturedAt"] = pd.to_datetime(df["capturedAt"])

### Split Data

In [35]:
date_counts = (
    df["capturedAt"]
    .dt.date
    .value_counts()
    .sort_index()
)
date_counts.index.min(), date_counts.index.max()

(datetime.date(2025, 1, 1), datetime.date(2025, 3, 22))

In [36]:
last_3_dates = (
    df["capturedAt"]
    .dt.normalize()
    .drop_duplicates()
    .sort_values()
    .tail(3)
)

# Split
df_train = df[~df["capturedAt"].dt.normalize().isin(last_3_dates)].copy()
df_val = df[df["capturedAt"].dt.normalize().isin(last_3_dates)].copy()

print("Train:", df_train["capturedAt"].dt.date.min(), "->", df_train["capturedAt"].dt.date.max())
print("Val  :", df_val["capturedAt"].dt.date.unique())
print("Rows train:", len(df_train))
print("Rows val:", len(df_val))

Train: 2025-01-01 -> 2025-03-19
Val  : [datetime.date(2025, 3, 21) datetime.date(2025, 3, 20)
 datetime.date(2025, 3, 22)]
Rows train: 263586
Rows val: 17561


## Date Time Extraction

In [37]:
def extract_date_features(_df):
    _df = _df.copy()

    # Create a binary feature indicating whether the date falls on a weekend
    _df["is_weekend"] = (
        _df["capturedAt"].dt.dayofweek >= 5
    ).astype(int)

    # Create a sequential day index starting from the first date in the dataset
    first_date = df["capturedAt"].dt.normalize().min()

    _df["day_index"] = (
        (_df["capturedAt"].dt.normalize() - first_date).dt.days + 1
    )

    # Apply cyclical encoding to the day of the month
    day = _df["capturedAt"].dt.day
    _df["day_sin"] = np.sin(2 * np.pi * day / 30)

    return _df

In [38]:
df_train = extract_date_features(df_train)
df_val = extract_date_features(df_val)

In [39]:
df_train["day_index"].value_counts().sort_index()

day_index
1        1
2      199
3     1079
5      948
6     2976
7      492
8     1837
10    1318
11     105
32      48
33    2229
34    1738
35    1736
36    2945
37    3371
38    1935
39    5339
40    4400
41    4233
42    1455
43    1811
44    4722
45    4128
46    4704
47    3774
48    5255
49    6617
50    5625
51    4979
52    4066
53    3375
54    4494
55    2860
56    4784
57    5916
58    7076
59    8348
60    8555
61    8668
62    8378
63    7389
64    7598
65    6789
66    7776
67    7822
68    7952
69    6285
70    7795
71    8282
72    7742
73    5596
74    5920
75    6338
76    5892
77    9367
78    8524
Name: count, dtype: int64

In [40]:
df_val["day_index"].value_counts()

day_index
80    7534
79    7327
81    2700
Name: count, dtype: int64

## Nominal Encoding

In [41]:
field_ids = ["shopId", "itemId", "modelId", "cat_id", "brand"]

In [42]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

In [43]:
df_train[field_ids] = encoder.fit_transform(df_train[field_ids]).astype(int)
df_val[field_ids] = encoder.transform(df_val[field_ids]).astype(int)

In [44]:
joblib.dump(encoder, "ordinal_encoder.pkl")

['ordinal_encoder.pkl']

### Scaler

In [45]:
field_ids = ["review_rating", "total_rating_count", "cmt_count", "shop_rating", "shop_response_rate", "shop_follower_count"]

In [46]:
scaler = MinMaxScaler()

In [47]:
df_train[field_ids] = scaler.fit_transform(df_train[field_ids])
df_val[field_ids] = scaler.transform(df_val[field_ids])

In [48]:
df_train[field_ids].describe()

,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count
count,263586.000000,263586.000000,263586.000000,263586.000000,263586.000000,263586.000000
mean,0.933163,0.005513,0.005264,0.896357,0.761633,0.019109
std,0.232523,0.018014,0.017653,0.078455,0.182931,0.062147
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.987500,0.000259,0.000261,0.877166,0.637363,0.001122
50%,0.997468,0.001035,0.000914,0.915474,0.802198,0.004303
75%,1.000000,0.003945,0.003593,0.945946,0.912088,0.017215
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [49]:
df_val[field_ids].describe()

,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count
count,17561.000000,17561.000000,17561.000000,17561.000000,17561.000000,17561.000000
mean,0.956209,0.006789,0.006429,0.880346,0.751967,0.035885
std,0.181846,0.018151,0.017847,0.089262,0.194093,0.110813
min,0.000000,0.000000,0.000000,0.219340,0.010989,0.000005
25%,0.988889,0.000259,0.000261,0.880508,0.582418,0.000754
50%,0.996364,0.001164,0.001110,0.888212,0.802198,0.004095
75%,1.000000,0.005885,0.005618,0.931182,0.923077,0.011240
max,1.000000,0.312314,0.315305,1.000000,1.000000,0.491436


In [50]:
joblib.dump(scaler, "minmax_scaler.pkl")

['minmax_scaler.pkl']

In [51]:
df_train.columns

Index(['capturedAt', 'shopId', 'itemId', 'modelId', 'price',
       'priceBeforeDiscount', 'promotionId', 'cat_id', 'raw_discount', 'brand',
       'is_free_shipping', 'is_pre_order', 'item_price_min', 'item_price_max',
       'review_rating', 'total_rating_count', 'cmt_count', 'shop_rating',
       'shop_response_rate', 'shop_follower_count', 'is_official_shop',
       'is_verified', 'is_weekend', 'day_index', 'day_sin'],
      dtype='object')

In [52]:
df_val.columns

Index(['capturedAt', 'shopId', 'itemId', 'modelId', 'price',
       'priceBeforeDiscount', 'promotionId', 'cat_id', 'raw_discount', 'brand',
       'is_free_shipping', 'is_pre_order', 'item_price_min', 'item_price_max',
       'review_rating', 'total_rating_count', 'cmt_count', 'shop_rating',
       'shop_response_rate', 'shop_follower_count', 'is_official_shop',
       'is_verified', 'is_weekend', 'day_index', 'day_sin'],
      dtype='object')

In [53]:
# Remove capturedAt
df_train = df_train.drop(columns=["capturedAt"]).copy()
df_val = df_val.drop(columns=["capturedAt"]).copy()

In [54]:
print(f"Train: {len(df_train):,} rows")
print(f"Validation: {len(df_val):,} rows")

Train: 263,586 rows
Validation: 17,561 rows


In [55]:
# Remove duplicates
df_train = df_train.drop_duplicates().reset_index(drop=True)
df_val = df_val.drop_duplicates().reset_index(drop=True)

In [56]:
print(f"Train: {len(df_train):,} rows")
print(f"Validation: {len(df_val):,} rows")

Train: 205,848 rows
Validation: 12,210 rows


In [57]:
df_train.to_csv("train_processed.csv", index=False)
df_val.to_csv("val_processed.csv", index=False)

### Final Form

In [58]:
df_train.columns

Index(['shopId', 'itemId', 'modelId', 'price', 'priceBeforeDiscount',
       'promotionId', 'cat_id', 'raw_discount', 'brand', 'is_free_shipping',
       'is_pre_order', 'item_price_min', 'item_price_max', 'review_rating',
       'total_rating_count', 'cmt_count', 'shop_rating', 'shop_response_rate',
       'shop_follower_count', 'is_official_shop', 'is_verified', 'is_weekend',
       'day_index', 'day_sin'],
      dtype='object')

In [59]:
df_train[['shopId', 'itemId', 'modelId', 'price', 'priceBeforeDiscount', 'promotionId', 'cat_id', 'raw_discount', 'brand']]

,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,raw_discount,brand
0,0,1307,4762,600.0,0.0,0,15,0,93
1,0,1307,4763,600.0,0.0,0,15,0,93
2,0,1307,4761,600.0,0.0,0,15,0,93
3,0,1307,4759,600.0,0.0,0,15,0,93
4,0,1307,4764,600.0,0.0,0,15,0,93
...,...,...,...,...,...,...,...,...,...
205843,215,1325,3602,1422.0,0.0,0,0,0,309
205844,215,1325,3601,855.0,0.0,0,0,0,309
205845,215,1325,3604,855.0,0.0,0,0,0,309
205846,215,1325,3605,1422.0,0.0,0,0,0,309


In [60]:
df_train[['is_free_shipping', 'is_pre_order', 'item_price_min', 'item_price_max', 'review_rating', 'total_rating_count', 'cmt_count', 'shop_rating']]

,is_free_shipping,is_pre_order,item_price_min,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating
0,0,0,600.0,600.0,0.998068,0.026775,0.025475,0.952010
1,0,0,600.0,600.0,0.998068,0.026775,0.025475,0.952010
2,0,0,600.0,600.0,0.998068,0.026775,0.025475,0.952010
3,0,0,600.0,600.0,0.998068,0.026775,0.025475,0.952010
4,0,0,600.0,600.0,0.998068,0.026775,0.025475,0.952010
...,...,...,...,...,...,...,...,...
205843,0,0,855.0,1755.0,0.000000,0.000000,0.000000,0.573884
205844,0,0,855.0,1755.0,0.000000,0.000000,0.000000,0.573884
205845,0,0,855.0,1755.0,0.000000,0.000000,0.000000,0.573884
205846,0,0,855.0,1755.0,0.000000,0.000000,0.000000,0.573884


In [61]:
df_train[['shop_response_rate', 'shop_follower_count', 'is_official_shop', 'is_verified', 'is_weekend', 'day_index', 'day_sin' ]]

,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_weekend,day_index,day_sin
0,0.758242,0.004433,0,0,0,3,0.587785
1,0.758242,0.004433,0,0,0,3,0.587785
2,0.758242,0.004433,0,0,0,3,0.587785
3,0.758242,0.004433,0,0,0,3,0.587785
4,0.758242,0.004433,0,0,0,3,0.587785
...,...,...,...,...,...,...,...
205843,0.527473,0.001552,0,0,0,76,-0.406737
205844,0.527473,0.001552,0,0,0,76,-0.406737
205845,0.527473,0.001552,0,0,0,76,-0.406737
205846,0.527473,0.001552,0,0,0,76,-0.406737
